In [2]:
# ============================================
# NOTEBOOK 3 - UNIT ECONOMICS E VALE DA MORTE
# ============================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from textwrap import fill

# --------------------------------------------
# 1️⃣ SIMULAÇÃO BASE - 36 MESES
# --------------------------------------------

meses = pd.date_range("2025-11", periods=36, freq="MS")

# Parâmetros ajustáveis (poderão vir de config futuramente)
params = {
    "taxa_crescimento_trafego": 0.10,
    "taxa_conversao_trial_pagante": 0.15,
    "churn": 0.04,
    "arpu": 97,
    "custo_ia_por_usuario": 5,
    "opex_fixo": 1800,
    "investimento_inicial": 2000
}

# Função principal de simulação
def simular_fluxo(params):
    base_trafego = 1000
    clientes = 50
    saldo = params["investimento_inicial"]
    registros = []
    for mes in meses:
        base_trafego *= (1 + params["taxa_crescimento_trafego"])
        novos_trials = base_trafego * 0.05
        novos_pagantes = novos_trials * params["taxa_conversao_trial_pagante"]
        clientes = clientes + novos_pagantes - (clientes * params["churn"])
        receita = clientes * params["arpu"]
        custo = clientes * params["custo_ia_por_usuario"]
        lucro = receita - custo - params["opex_fixo"]
        saldo += lucro
        registros.append({
            "mes": mes,
            "clientes_ativos": clientes,
            "receita": receita,
            "custo": custo,
            "lucro": lucro,
            "saldo_caixa": saldo
        })
    return pd.DataFrame(registros)

df = simular_fluxo(params)

# --------------------------------------------
# 2️⃣ CÁLCULO DE UNIT ECONOMICS
# --------------------------------------------

df["CAC"] = 500  # custo médio de aquisição por cliente
df["LTV"] = params["arpu"] / params["churn"]
df["LTV_CAC_ratio"] = df["LTV"] / df["CAC"]
df["payback_meses"] = (df["CAC"] / params["arpu"]).round(1)

# KPIs finais
ltv_final = df["LTV"].iloc[-1]
cac_final = df["CAC"].iloc[-1]
ratio_final = df["LTV_CAC_ratio"].iloc[-1]
payback_final = df["payback_meses"].iloc[-1]

print("\n📊 MÉTRICAS UNIT ECONOMICS (Resumo Final)\n")
print(f"LTV (Lifetime Value): R$ {ltv_final:,.2f}")
print(f"CAC (Custo de Aquisição de Cliente): R$ {cac_final:,.2f}")
print(f"Relação LTV/CAC: {ratio_final:.2f}x  ➜ {'Sustentável ✅' if ratio_final >= 3 else 'Crítico ⚠️'}")
print(f"Payback médio por cliente: {payback_final} meses")

# --------------------------------------------
# 3️⃣ GRÁFICO 1 - LTV vs CAC ao longo do tempo
# --------------------------------------------

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["mes"], y=df["LTV"],
    mode="lines", name="LTV (Valor vitalício do cliente)",
    line=dict(color="green", width=3)
))
fig.add_trace(go.Scatter(
    x=df["mes"], y=df["CAC"],
    mode="lines", name="CAC (Custo de aquisição)",
    line=dict(color="red", width=3)
))
fig.update_layout(
    title="Evolução do LTV e CAC ao longo do tempo",
    xaxis_title="Mês",
    yaxis_title="Valor em R$",
    legend_title="Indicadores",
    hovermode="x unified"
)
fig.show()

print(fill("""
💬 INTERPRETAÇÃO:
Este gráfico mostra quanto cada cliente gera de receita (LTV) comparado ao custo de adquiri-lo (CAC).
O eixo X representa o tempo (em meses) e o eixo Y os valores monetários.
A curva verde deve ficar consistentemente acima da vermelha para indicar sustentabilidade.
Regra de ouro: LTV/CAC > 3x é saudável. Menor que isso sugere modelo insustentável.
""", width=100))

# --------------------------------------------
# 4️⃣ GRÁFICO 2 - Saldo de Caixa e “Vale da Morte”
# --------------------------------------------

min_caixa = df["saldo_caixa"].min()
mes_min = df.loc[df["saldo_caixa"].idxmin(), "mes"]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df["mes"], y=df["saldo_caixa"],
    mode="lines+markers", name="Saldo de Caixa",
    line=dict(color="blue", width=3)
))
fig2.add_hline(y=0, line_dash="dot", line_color="gray")
fig2.add_annotation(
    x=mes_min, y=min_caixa,
    text=f"Vale da Morte: R$ {min_caixa:,.0f} ({mes_min.strftime('%b/%Y')})",
    showarrow=True, arrowhead=2, ax=0, ay=-40, font=dict(color="red")
)
fig2.update_layout(
    title="Evolução do Caixa e Identificação do 'Vale da Morte'",
    xaxis_title="Mês",
    yaxis_title="Saldo de Caixa (R$)",
    hovermode="x unified"
)
fig2.show()

print(fill("""
💬 INTERPRETAÇÃO:
O “Vale da Morte” é o ponto em que o saldo de caixa chega ao seu valor mais baixo.
No gráfico acima, a linha azul representa o saldo acumulado mês a mês.
A linha pontilhada cinza é o nível zero (sem caixa).
A anotação vermelha marca o ponto mais crítico — se o negócio não tiver capital suficiente até ali, quebra.
""", width=100))

# --------------------------------------------
# 5️⃣ GRÁFICO 3 - Burn Rate Mensal e Runway
# --------------------------------------------

df["burn_rate"] = -df["lucro"].where(df["lucro"] < 0, 0)
media_burn = df["burn_rate"].mean()
runway = (df["saldo_caixa"].iloc[-1] / media_burn) if media_burn > 0 else np.nan

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=df["mes"], y=df["burn_rate"],
    name="Burn Rate (gasto mensal líquido)", marker_color="orange"
))
fig3.update_layout(
    title="Burn Rate Mensal — ritmo de queima de caixa",
    xaxis_title="Mês",
    yaxis_title="Valor negativo (R$)",
)
fig3.show()

print(fill(f"""
💬 INTERPRETAÇÃO:
O gráfico mostra quanto dinheiro é queimado mensalmente.
As barras representam meses em que há prejuízo operacional (lucro negativo).
Burn Rate médio: R$ {media_burn:,.2f}/mês
Runway estimado: {runway:.1f} meses de sobrevida no ritmo atual.
""", width=100))

# --------------------------------------------
# 6️⃣ ALERTAS AUTOMÁTICOS
# --------------------------------------------

print("\n🚨 ALERTAS FINANCEIROS AUTOMÁTICOS\n")

if ratio_final < 3:
    print("⚠️ LTV/CAC abaixo de 3x: reveja estratégia de retenção e aquisição.")
if min_caixa < 0:
    print("⚠️ Caixa negativo detectado: risco de quebra. Considere aporte de capital.")
if runway < 6:
    print("⚠️ Runway menor que 6 meses: ajuste custos ou busque investimento urgente.")
if media_burn > 10000:
    print("⚠️ Burn rate elevado: revise OPEX e eficiência operacional.")
else:
    print("✅ Nenhum alerta crítico. Modelo sustentável no curto prazo.")



📊 MÉTRICAS UNIT ECONOMICS (Resumo Final)

LTV (Lifetime Value): R$ 2,425.00
CAC (Custo de Aquisição de Cliente): R$ 500.00
Relação LTV/CAC: 4.85x  ➜ Sustentável ✅
Payback médio por cliente: 5.2 meses


 💬 INTERPRETAÇÃO: Este gráfico mostra quanto cada cliente gera de receita (LTV) comparado ao custo
de adquiri-lo (CAC). O eixo X representa o tempo (em meses) e o eixo Y os valores monetários. A
curva verde deve ficar consistentemente acima da vermelha para indicar sustentabilidade. Regra de
ouro: LTV/CAC > 3x é saudável. Menor que isso sugere modelo insustentável.


 💬 INTERPRETAÇÃO: O “Vale da Morte” é o ponto em que o saldo de caixa chega ao seu valor mais baixo.
No gráfico acima, a linha azul representa o saldo acumulado mês a mês. A linha pontilhada cinza é o
nível zero (sem caixa). A anotação vermelha marca o ponto mais crítico — se o negócio não tiver
capital suficiente até ali, quebra.


 💬 INTERPRETAÇÃO: O gráfico mostra quanto dinheiro é queimado mensalmente. As barras representam
meses em que há prejuízo operacional (lucro negativo). Burn Rate médio: R$ 0.00/mês Runway estimado:
nan meses de sobrevida no ritmo atual.

🚨 ALERTAS FINANCEIROS AUTOMÁTICOS

✅ Nenhum alerta crítico. Modelo sustentável no curto prazo.
